# Homomorphic mode recovery via kernel: SIMD version

Implements Sections 4.1 and 4.7 of the dissertation: a single packed ciphertext,
padding to `N = 2^ceil(log2 n)`, and the mask applied to the **output** of the
polynomial.

Contents:

1. Data and plaintext reference functions
2. Plaintext sweep over `(sigma, d)`, measuring `eps`, `Delta_S` and the condition `n*eps < Delta_S/2`
3. FHE run with the chosen parameters
4. Exact / approximate / FHE comparison

Cells 1 and 2 run in seconds and do not need OpenFHE. Use them to choose the
parameters before spending time on FHE.

In [ ]:
import math
import numpy as np
from numpy.polynomial.chebyshev import Chebyshev

# ============================================================
# DATA
# ============================================================
dados = np.array(
    [10.0] * 20                                                    # mode: 20 copies
    + [20.03, 19.97, 20.08, 19.92, 20.12, 19.88,
       20.05, 19.95, 20.10, 19.90, 20.02]                          # cluster ~20
    + [29.94, 30.11, 29.89, 30.04, 29.96]                          # cluster ~30
)

n = len(dados)
N = 2 ** math.ceil(math.log2(n))

valores = np.unique(dados)
freqs = {v: int((dados == v).sum()) for v in valores}
v_estrela = max(freqs, key=freqs.get)
f_m = freqs[v_estrela]
delta_f = f_m - max(f for v, f in freqs.items() if v != v_estrela)
delta_min = min(abs(a - b) for a in valores for b in valores if a != b)
delta_max = dados.max() - dados.min()

print(f"n = {n},  N = {N},  valores distintos = {len(valores)}")
print(f"moda v* = {v_estrela},  f_m = {f_m},  Delta_f = {delta_f}")
print(f"Delta_min = {delta_min:.4f},  Delta_max = {delta_max:.4f}")
print(f"racio Delta_max/Delta_min = {delta_max/delta_min:.0f}")


# ============================================================
# REFERENCE FUNCTIONS (plaintext)
# ============================================================
def matriz_u(x, sigma):
    """u_ij = ((x_i - x_j)/sigma)^2"""
    return ((x[:, None] - x[None, :]) / sigma) ** 2


def scores_exatos(x, sigma):
    """S_i = sum_j K((x_i-x_j)/sigma),  K(t) = 1/(1+t^2)"""
    return (1.0 / (1.0 + matriz_u(x, sigma))).sum(axis=1)


def polinomio(R, d):
    """Chebyshev approximation of F(u) = 1/(1+u) on [0, R]."""
    return Chebyshev.interpolate(lambda u: 1.0 / (1.0 + u), d, domain=[0.0, R])


def eps_uniforme(R, d, amostras=200_001):
    """eps = ||F - P_d||_inf on [0, R], measured numerically."""
    P = polinomio(R, d)
    us = np.linspace(0.0, R, amostras)
    return float(np.abs(1.0 / (1.0 + us) - P(us)).max())


def scores_aprox(x, sigma, d, termo_k0=True):
    """Scores with the polynomial, in exact arithmetic (no CKKS noise)."""
    R = ((x.max() - x.min()) / sigma) ** 2
    K = polinomio(R, d)(matriz_u(x, sigma))
    if termo_k0:
        np.fill_diagonal(K, 1.0)   # the pair (i,i) contributes exactly K(0)=1
    return K.sum(axis=1)


def gap_de_scores(sc, x):
    """Delta_S at the level of VALUES (Remark 4.2)."""
    por_valor = {v: sc[np.flatnonzero(x == v)[0]] for v in np.unique(x)}
    s_moda = por_valor[v_estrela]
    s_rival = max(s for v, s in por_valor.items() if v != v_estrela)
    return s_moda - s_rival


def rho_bernstein(R):
    """Bernstein ellipse parameter for the pole u = -1, mapped from [0,R]."""
    s0 = 1.0 + 2.0 / R
    return s0 + math.sqrt(s0 * s0 - 1.0)


def sigma_teorema_4_6():
    """Solves f_m > f_y + (n - f_y) K(Delta/sigma) exactly for sigma.
    With K(t) = 1/(1+t^2) this gives  sigma < Delta_min * sqrt(Delta_f/(n - f_m)).
    SUFFICIENT condition."""
    return delta_min * math.sqrt(delta_f / (n - f_m))


print(f"\nLimite do Teorema 4.6:  sigma < {sigma_teorema_4_6():.5f}")
print("(suficiente, nao necessario; ver varrimento abaixo)")

## 2. Plaintext sweep over `(sigma, d)`

For each pair the sweep reports:

- `R = (Delta_max/sigma)^2`: the approximation domain (Section 4.6)
- `eps`: the measured uniform error
- `Delta_S`: the exact score gap (Section 4.8)
- whether `n*eps < Delta_S/2`, the condition of Theorem 4.12
- whether the approximate argmax falls in the modal class

The last column shows where the protocol still works although the sufficient
condition fails.

In [ ]:
classe_modal = set(np.flatnonzero(dados == v_estrela).tolist())

print(f"{'sigma':>6} {'d':>4} {'R':>10} {'eps':>10} {'n*eps':>9} "
      f"{'Delta_S':>9} {'cond':>6} {'argmax':>7} {'S(v*)':>8}")
print("-" * 78)

linhas = []
for sigma in (1.0, 2.0, 3.0, 5.0):
    R = (delta_max / sigma) ** 2
    exatos = scores_exatos(dados, sigma)
    dS = gap_de_scores(exatos, dados)
    for d in (15, 31, 63, 127):
        eps = eps_uniforme(R, d)
        ap = scores_aprox(dados, sigma, d)
        ok_cond = n * eps < dS / 2
        ok_argmax = int(np.argmax(ap)) in classe_modal
        linhas.append((sigma, d, R, eps, dS, ok_cond, ok_argmax))
        print(f"{sigma:6.1f} {d:4d} {R:10.1f} {eps:10.2e} {n*eps:9.3f} "
              f"{dS:9.3f} {'SIM' if ok_cond else 'nao':>6} "
              f"{'SIM' if ok_argmax else 'NAO':>7} {ap[0]:8.4f}")

print(f"\nScore exato da moda (sigma=1): {scores_exatos(dados, 1.0)[0]:.4f}")
print(f"Score exato da moda (sigma=3): {scores_exatos(dados, 3.0)[0]:.4f}")

### Checking the Bernstein bound

Section 4.6 states `||F - P_d||_inf <= C * rho^(-d)`. This cell checks that the
measured decay has the rate `rho` predicted by Proposition 4.23.

In [ ]:
for sigma in (1.0, 3.0):
    R = (delta_max / sigma) ** 2
    rho = rho_bernstein(R)
    graus = np.arange(8, 80, 4)
    eps = np.array([eps_uniforme(R, int(d), 50_001) for d in graus])

    # slope of log(eps) against d;  predicted: -log(rho)
    declive = np.polyfit(graus, np.log(eps), 1)[0]

    print(f"sigma = {sigma}   R = {R:.1f}   rho = {rho:.4f}")
    print(f"  -log(rho) previsto : {-math.log(rho):+.5f}")
    print(f"  declive medido     : {declive:+.5f}")
    print(f"  racio              : {declive / -math.log(rho):.3f}")
    print(f"  d para eps < 0.01  : {math.ceil(math.log(1/0.01)/math.log(rho))}\n")

## 3. FHE run: SIMD version

With `N = 64`, this version uses 63 rotations, 1 rotation key, 1 output ciphertext
and 1 decryption. The `k=0` term enters as an exact constant.

The loop runs over shifts `k`, not points `i`. A rotation by `k` aligns slot `i`
with the value `x_{i+k}`, so a single polynomial evaluation handles every pair at
distance `k` at once.

`k` runs up to `N-1`, not `n-1`: the rotation is cyclic over `N` slots, so every
shift of the buffer is needed to cover all pairs.

In [ ]:
import openfhe

# ---- parameters chosen from the sweep above -------------------------------
SIGMA = 3.0
GRAU = 31
# ---------------------------------------------------------------------------

x_esc = dados / SIGMA
R_fhe = (x_esc.max() - x_esc.min()) ** 2

# depth: square (1) + Chebyshev (ceil(log2(d+1))+1) + mask (1)
prof = 1 + (math.ceil(math.log2(GRAU + 1)) + 1) + 1

print(f"sigma = {SIGMA}   grau = {GRAU}   R = {R_fhe:.1f}")
print(f"profundidade do circuito = {prof}")
print(f"eps previsto = {eps_uniforme(R_fhe, GRAU):.3e}\n")

params = openfhe.CCParamsCKKSRNS()
params.SetMultiplicativeDepth(prof + 2)   # margin
params.SetScalingModSize(50)              # precision: scores ~ n, gap ~ 1
params.SetBatchSize(N)

cc = openfhe.GenCryptoContext(params)
cc.Enable(openfhe.PKESchemeFeature.PKE)
cc.Enable(openfhe.PKESchemeFeature.LEVELEDSHE)
cc.Enable(openfhe.PKESchemeFeature.ADVANCEDSHE)

keys = cc.KeyGen()
cc.EvalMultKeyGen(keys.secretKey)
cc.EvalRotateKeyGen(keys.secretKey, [1])
print("chaves de rotacao: 1")

# Padding slots must hold a value INSIDE [0, R]. With 0.0, the (real, padding)
# pairs give u = x^2 far above R and the Chebyshev polynomial blows up
# (P_31(100.7) ~ -3.5e20), which ruins the precision of the whole ciphertext because
# the CKKS scale is global. The mask only zeroes those terms after the polynomial.
PAD = float(x_esc[0])

enc = cc.Encrypt(keys.publicKey,
                 cc.MakeCKKSPackedPlaintext(list(x_esc) + [PAD] * (N - n)))


def kernel_func(u):
    return 1.0 / (1.0 + max(0.0, u))


def mascara(vals, ct=None):
    """Plaintext at the ciphertext's level (avoids a level mismatch in EvalMult)."""
    if ct is not None:
        try:
            return cc.MakeCKKSPackedPlaintext(vals, 1, ct.GetLevel(), None, N)
        except Exception:
            pass
    return cc.MakeCKKSPackedPlaintext(vals)


# ---- SIMD loop: one shift at a time ---------------------------------------
acc = None
n_rot = n_cheb = 0
rot = enc          # invariant: at the start of step k, rot = enc rotated by k-1

for k in range(1, N):
    # always rotate by 1, even when the step is skipped, to keep the invariant.
    # This way the k=1 rotation key suffices, instead of one key per shift.
    rot = cc.EvalRotate(rot, 1);       n_rot += 1

    # pair (i, i+k mod N) is valid iff both indices are real
    m = [1.0 if (i < n and (i + k) % N < n) else 0.0 for i in range(N)]
    if not any(m):
        continue
    diff = cc.EvalSub(enc, rot)
    d2 = cc.EvalMult(diff, diff)
    kv = cc.EvalChebyshevFunction(kernel_func, d2, 0.0, R_fhe, GRAU)
    n_cheb += 1

    kv = cc.EvalMult(kv, mascara(m, kv))        # mask on the OUTPUT of the polynomial
    acc = kv if acc is None else cc.EvalAdd(acc, kv)

    if k % 16 == 0:
        print(f"  deslocamento {k}/{N - 1}")

# k=0 term: the pair (i,i) contributes exactly K(0)=1, without the polynomial
acc = cc.EvalAdd(acc, mascara([1.0] * n + [0.0] * (N - n), acc))

print(f"\nrotacoes: {n_rot}   avaliacoes de Chebyshev: {n_cheb}")

# ---- a single decryption --------------------------------------------------
pt = cc.Decrypt(keys.secretKey, acc)
pt.SetLength(n)
scores_fhe = np.array(pt.GetRealPackedValue())

In [ ]:
# ============================================================
# 4. COMPARISON
# ============================================================
exatos = scores_exatos(dados, SIGMA)
aprox = scores_aprox(dados, SIGMA, GRAU)

print(f"{'i':>4} {'x_i':>8} {'exato':>10} {'polinomio':>11} {'FHE':>10} {'|dif|':>9}")
print("-" * 56)
for i in np.argsort(scores_fhe)[::-1][:12]:
    print(f"{i:4d} {dados[i]:8.2f} {exatos[i]:10.4f} {aprox[i]:11.4f} "
          f"{scores_fhe[i]:10.4f} {abs(exatos[i]-scores_fhe[i]):9.4f}")

dS = gap_de_scores(exatos, dados)
eps = eps_uniforme(R_fhe, GRAU)
eta = float(np.abs(aprox - scores_fhe).max())   # observed CKKS noise

print(f"\n--- Teorema 4.16 ---------------------------------------")
print(f"Delta_S            = {dS:.4f}")
print(f"n*eps              = {n*eps:.4f}")
print(f"eta observado      = {eta:.2e}")
print(f"n*eps + eta        = {n*eps + eta:.4f}")
print(f"Delta_S / 2        = {dS/2:.4f}")
print(f"condicao satisfeita: {n*eps + eta < dS/2}")

melhor = int(np.argmax(scores_fhe))
arg_fhe = set(np.flatnonzero(
    scores_fhe > scores_fhe.max() - 10 * max(eta, 1e-9)).tolist())

print(f"\n--- Resultado ------------------------------------------")
print(f"moda recuperada    = {dados[melhor]}   (correto: {v_estrela})")
print(f"classe modal M     = {len(classe_modal)} indices")
print(f"argmax dentro de M : {arg_fhe <= classe_modal}")
print(f"desvio entre copias: {np.ptp(scores_fhe[list(classe_modal)]):.3e}")
print("(Remark 4.17: o ruido CKKS quebra os empates exatos)")